# ReAct Loop via LangChain

## Problem Statement
Wire a ReAct agent for DevContext AI that reasons over two tools: search_codebase and search_docs.

## My Approach
I modeled the ReAct loop mentally before touching code: user question → LLM decides which tool to call → tool returns a result → LLM reads it and decides whether it needs another tool or has enough to answer. 

The agent's "memory" is just the growing messages list - each tool result gets appended, and the LLM sees the full history on every reasoning step. I built two simple dict-backed tools to simulate what will eventually be ChromaDB queries, then wired them to create_react_agent and ran all 8 queries to see the loop in action.

## What I Learned
Model size matters more than prompt engineering for ReAct. The 8b model looped endlessly - calling the same tool 4 times with identical inputs despite getting the correct result back - because it couldn't recognize it already had enough to answer.
Switching to 70b fixed it immediately, no prompt changes needed. 

The other big lesson: tool docstrings are the tool selection logic. Vague descriptions like "use for how-to questions" aren't enough - the agent picks tools based on semantic similarity between the query and the docstring, so precision there directly determines whether it calls the right tool.

## Where I Got Stuck
search_docs was silently failing on Query 1 ("Is there any documentation on the payment flow?"). The doc key in DOCS is payment_flow (underscore) but the agent was sending queries like "payment flow documentation" (spaces, extra word). The naive substring check "payment_flow" in "payment flow documentation" returns False because of the underscore-vs-space mismatch. 

Every tool retry came back empty, the agent hit the recursion limit, and printed "Sorry, need more steps." The fix was normalizing underscores to spaces and switching to token-overlap matching so partial matches work correctly.

## What I'd Do Differently
Start with a fuzzy/token-overlap matcher from day one instead of the naive substring check - it broke on the very first real query. Also, I'd add a tool_call_count counter to the trace output immediately; I had to count steps manually to spot the 8b looping behavior. Going forward for custom tools, I'll write the docstring first and ask myself "would the LLM know exactly when to call this?" before writing any tool logic.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
load_dotenv()

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [17]:
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0) 


In [5]:
# Simulated codebase index
CODEBASE = {
    "process_payment": "Calls validate_card(card_number), then calls charge_gateway(). Returns dict {status, transaction_id}.",
    "validate_card": "Checks Luhn algorithm on card_number. Returns True/False. Raises CardError if card_number is None.",
    "charge_gateway": "POSTs to external gateway API. Retries 3x on timeout. Returns gateway response dict.",
}

# Simulated docs index
DOCS = {
    "payment_flow": "Payment flow: frontend → POST /pay → process_payment() → validate_card() → charge_gateway() → respond. Card errors surface as 400. Gateway timeouts surface as 503.",
    "onboarding": "New engineers: read CONTRIBUTING.md first. Payment module owned by @payments-team.",
}

QUERIES = [
    # Single-source: codebase only
    "What does the validate_card function do?",

    # Single-source: docs only
    "Is there any documentation on the payment flow?",

    # Two-source: requires both tools
    "The process_payment function calls validate_card — what does validate_card do, and is there any documentation on the payment flow?",

    # Not-found case: neither source has this
    "What does the refund_crypto function do?",

    # Not-found in docs, exists in code
    "Is there documentation on the charge_gateway function?",

    # Exists in docs, not in code as a function
    "Who owns the payment module and where should a new engineer start?",

    # Multi-hop: needs code result to form the next question
    "process_payment calls something to check the card — what does that function raise if the card number is None?",

    # Ambiguous: could match both sources
    "How does payment processing work end to end?",
]

In [ ]:
@tool
def search_codebase(query: str) -> str:
    """Search the codebase index for functions, classes, and callers. Use for code-related questions."""
    # Simulated index — in DevContext AI this queries ChromaDB
    query_lc = query.lower()
    for key, val in CODEBASE.items():
        key_lc = key.lower()
        if key_lc in query_lc or query_lc in key_lc:
            return val
    return f"No matching code found for: {query}"

@tool
def search_docs(query: str) -> str:
    """Search internal documentation and README files. Use for how-to and architecture questions."""
    query_lc = query.lower()
    for key, val in DOCS.items():
        key_lc = key.lower()
        # OPTIMIZED: Check if key is in query OR query is in key
        if key_lc in query_lc or query_lc in key_lc:
            return val
    return f"No matching code found for: {query}"


In [23]:
agent = create_react_agent(llm 
                           , tools = [search_codebase , search_docs] 
                           , prompt="You are DevContext AI - a developer assistant for onboarding and PR impact analysis. When calling tools, you MUST ensure the arguments are valid JSON. Double-check that every opening brace { has a matching closing brace } and all strings are quoted.")

C:\Users\khush\AppData\Local\Temp\ipykernel_16264\2625795858.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm


In [21]:
#Test

try:
    print("--Running ReAct agent...\n")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": QUERIES[0]}]}
        , config={"recursion_limit": 10}
    )
    #Final answer - last message is final answer

    print("Query : " , QUERIES[0])
    final_ans=result["messages"][-1]
    print("Answer : ", final_ans.content)
    print("Reasoning Trace :-\n")
    for (i,msg) in enumerate(result["messages"]) :
        role=msg.__class__.__name__ #this will return AIMessage, HumanMessage, or ToolMessage just to see talking to itself versus calling tools
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[Step {i}] TOOL CALL: {tc['name']}({tc['args']})")
        elif hasattr(msg, 'content') and msg.content and role != 'HumanMessage':
            preview = str(msg.content)[:120]
            print(f"[Step {i}] {role}: {preview}")
except GraphRecursionError:
    print("Agent could not complete reasoning in the allowed steps. Try a more specific question.")

--Running ReAct agent...

Query :  What does the validate_card function do?
Answer :  The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the card number is valid and `False` otherwise. If the card number is `None`, it raises a `CardError`.
Reasoning Trace :-

[Step 1] TOOL CALL: search_codebase({'query': 'validate_card function'})
[Step 2] ToolMessage: Checks Luhn algorithm on card_number. Returns True/False. Raises CardError if card_number is None.
[Step 3] AIMessage: The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the c


In [28]:
print("--Running ReAct agent...\n")
for q_idx, query in enumerate(QUERIES) :
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": query}]}
            , config={"recursion_limit": 10}
        )
        #Final answer - last message is final answer

        print(f"\nQuery [{q_idx}]: {query}")
        final_ans=result["messages"][-1]
        print("Answer : ", final_ans.content)
        print()
        print("Reasoning Trace :-\n")
        for (i,msg) in enumerate(result["messages"]) :
            #this will return AIMessage, HumanMessage, or ToolMessage just to see talking to itself versus calling tools
            role=msg.__class__.__name__ 
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[Step {i}] TOOL CALL: {tc['name']}({tc['args']})")
            elif hasattr(msg, 'content') and msg.content and role != 'HumanMessage':
                preview = str(msg.content)[:120]
                print(f"[Step {i}] {role}: {preview}")
    except Exception as e :
        print("Agent encountered problem :", e)
        continue

--Running ReAct agent...


Query [0]: What does the validate_card function do?
Answer :  The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the card number is valid and `False` otherwise. If the card number is `None`, it raises a `CardError`.

Reasoning Trace :-

[Step 1] TOOL CALL: search_codebase({'query': 'validate_card function'})
[Step 2] ToolMessage: Checks Luhn algorithm on card_number. Returns True/False. Raises CardError if card_number is None.
[Step 3] AIMessage: The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the c

Query [1]: Is there any documentation on the payment flow?
Answer :  It seems that there is no documentation available on the payment flow. You may want to try searching for related topics or checking with the development team for more information.

Reasoning Trace :-

[Step 1] TOOL CALL: search_docs({'query': 'payment flow documentat

## Refactored Solution

In [29]:
def _fuzzy_match(query: str, index: dict) -> str | None:
    """
    Match query against index keys using token overlap.
    Normalizes underscores → spaces so 'payment_flow' matches 'payment flow'.
    Returns the value for the best-matching key, or None if no match.
    """
    query_tokens = set(query.lower().replace("_", " ").split())
    best_key, best_score = None, 0
    for key, val in index.items():
        key_tokens = set(key.lower().replace("_", " ").split())
        overlap = len(query_tokens & key_tokens)
        if overlap > best_score:
            best_score, best_key = overlap, key
    return index[best_key] if best_key and best_score >= 1 else None


@tool
def search_codebase(query: str) -> str:
    """
    Search the codebase index for functions, classes, and callers.
    Use for code-related questions: what a function does, what it calls, what it raises.
    Input: a function name or short description of what you're looking for.
    """
    result = _fuzzy_match(query, CODEBASE)
    return result if result else f"No matching code found for: '{query}'"


@tool
def search_docs(query: str) -> str:
    """
    Search internal documentation, README files, and architecture guides.
    Use for how-to questions, architecture overviews, team ownership, and onboarding guides.
    Input: a topic name or short description of what documentation you need.
    """
    result = _fuzzy_match(query, DOCS)
    return result if result else f"No matching documentation found for: '{query}'"

In [30]:
SYSTEM_PROMPT = (
    "You are DevContext AI — a developer assistant for codebase onboarding and PR impact analysis. "
    "Use search_codebase for questions about functions, classes, and code behavior. "
    "Use search_docs for architecture, flow, and team ownership questions. "
    "When a query needs both, call both tools before answering. "
    "If a tool returns 'No matching', report it clearly — do not guess or invent information."
)

In [ ]:
agent = create_agent(
    llm, 
    tools=[search_codebase, search_docs], 
    system_prompt=SYSTEM_PROMPT 
)

In [38]:
def print_trace(messages: list) -> None:
    """Print a clean ReAct trace from the messages list."""
    tool_call_count = 0
    for i, msg in enumerate(messages):
        role = msg.__class__.__name__
        if role == "HumanMessage":
            continue
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_call_count += 1
                print(f"  [Step {i}] TOOL CALL: {tc['name']}({tc['args']})")
        elif hasattr(msg, "content") and msg.content:
            preview = str(msg.content)[:150]
            print(f"  [Step {i}] {role}: {preview}")
    print(f"  Total tool calls: {tool_call_count}")


print("Running ReAct agent over all queries...\n")
print("=" * 70)

for q_idx, query in enumerate(QUERIES):
    print(f"\nQuery [{q_idx}]: {query}")
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config={"recursion_limit": 10}
        )
        final_answer = result["messages"][-1].content
        print(f"Answer: {final_answer}")
        print("\nTrace:")
        print_trace(result["messages"])
    except GraphRecursionError:
        print("ERROR: Agent hit recursion limit — tool loop detected. Check tool docstrings.")
    except Exception as e:
        print(f"ERROR: {e}")
    print("-" * 70)

Running ReAct agent over all queries...


Query [0]: What does the validate_card function do?
Answer: The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the card number is valid and `False` otherwise. If the card number is `None`, it raises a `CardError`.

Trace:
  [Step 1] TOOL CALL: search_codebase({'query': 'validate_card function'})
  [Step 2] ToolMessage: Checks Luhn algorithm on card_number. Returns True/False. Raises CardError if card_number is None.
  [Step 3] AIMessage: The `validate_card` function checks if a given card number is valid using the Luhn algorithm. It returns `True` if the card number is valid and `False
  Total tool calls: 1
----------------------------------------------------------------------

Query [1]: Is there any documentation on the payment flow?
ERROR: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 't